# Error Analysis for SAM-FC (ACIIDS 2027)
This notebook reproduces the error analysis reported in Section 5.3 of the paper:
- **Table 4**: Test performance by claim length stratification
- **Figure 4**: Cross-domain confusion matrices on ViFactCheck & ViWikiFC
- **Table 5**: Cross-domain conflict-slot attribution on Refuted claims
- **Case Study**: Relational/action reversal analysis (Case 1 in Section 5.3.2)


In [ ]:
import json
from collections import Counter
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from tabulate import tabulate

VFC_PATH = Path("../outputs/predictions/vifactcheck_test_predictions_ensemble.json")
WIKI_PATH = Path("../outputs/predictions/viwikifc_test_predictions_ensemble.json")

with open(VFC_PATH, "r", encoding="utf-8") as f:
    vfc_preds = json.load(f)
with open(WIKI_PATH, "r", encoding="utf-8") as f:
    wiki_preds = json.load(f)

print(f"Loaded {len(vfc_preds)} ViFactCheck predictions and {len(wiki_preds)} ViWikiFC predictions.")


## 1. Table 4: Claim Length Stratification
Evaluates Macro-F1 and Accuracy across Short (<=20 words), Medium (21-35 words), and Long (>35 words).

In [ ]:
def stratify(preds, ds_name):
    short = [x for x in preds if len(x["claim"].split()) <= 20]
    med = [x for x in preds if 21 <= len(x["claim"].split()) <= 35]
    long = [x for x in preds if len(x["claim"].split()) > 35]
    rows = []
    for name, items in [("Short (<20)", short), ("Medium (20-35)", med), ("Long (>35)", long)]:
        acc = accuracy_score([x["gold_verdict"] for x in items], [x["pred_verdict"] for x in items]) * 100
        f1 = f1_score([x["gold_verdict"] for x in items], [x["pred_verdict"] for x in items], average="macro") * 100
        rows.append([ds_name, name, len(items), f"{len(items)/len(preds)*100:.1f}%", f"{acc:.2f}", f"{f1:.2f}"])
    return rows

rows = stratify(vfc_preds, "ViFactCheck") + stratify(wiki_preds, "ViWikiFC")
print(tabulate(rows, headers=["Dataset", "Length", "Count", "Ratio", "Acc.", "Macro-F1"], tablefmt="github"))


## 2. Table 5: Cross-Domain Conflict Breakdown
Detailed breakdown across Numeric, Negation, Entity, and Temporal conflicts on Refuted claims.

In [ ]:
def conflict_table(preds, ds_name):
    refuted = [x for x in preds if x["gold_verdict"].capitalize() == "Refuted"]
    rows = []
    for ctype in ["NUMERIC", "NEGATION", "ENTITY", "TEMPORAL"]:
        items = [x for x in refuted if x["gold_conflict"].upper() == ctype]
        if not items: continue
        v_rec = sum(1 for x in items if x["pred_verdict"].capitalize() == "Refuted") / len(items) * 100
        s_acc = sum(1 for x in items if x.get("is_conflict_correct", False)) / len(items) * 100
        rows.append([ds_name, ctype.capitalize(), len(items), f"{v_rec:.1f}%", f"{s_acc:.1f}%"])
    return rows

c_rows = conflict_table(vfc_preds, "ViFactCheck") + conflict_table(wiki_preds, "ViWikiFC")
print(tabulate(c_rows, headers=["Dataset", "Type", "Count", "Verdict Recall", "Slot Acc."], tablefmt="github"))


## 3. Figure 4: Cross-Domain Confusion Matrices
Visualizes raw counts and recall percentages across Supported, Refuted, and NEI classes.

In [ ]:
labels = ["Supported", "Refuted", "NEI"]
cm_vfc = confusion_matrix([p["gold_verdict"] for p in vfc_preds], [p["pred_verdict"] for p in vfc_preds], labels=labels)
cm_wiki = confusion_matrix([p["gold_verdict"] for p in wiki_preds], [p["pred_verdict"] for p in wiki_preds], labels=labels)
rec_vfc = cm_vfc.astype("float") / cm_vfc.sum(axis=1, keepdims=True) * 100
rec_wiki = cm_wiki.astype("float") / cm_wiki.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), dpi=150)
for ax, rec, cm, title in zip(axes, [rec_vfc, rec_wiki], [cm_vfc, cm_wiki], ["(a) ViFactCheck", "(b) ViWikiFC"]):
    im = ax.imshow(rec, cmap="Blues", vmin=0, vmax=100)
    ax.set_xticks(range(3)); ax.set_xticklabels(labels)
    ax.set_yticks(range(3)); ax.set_yticklabels(labels)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    for i in range(3):
        for j in range(3):
            c = "white" if rec[i, j] > 50 else "black"
            ax.text(j, i, f"{cm[i, j]}
({rec[i, j]:.1f}%)", ha="center", va="center", color=c, fontsize=10)
plt.tight_layout()
plt.show()
